# International Hotel Review Analytics: Predicting Excellent Guest Satisfaction

## Project overview
This project uses a multi-table hotel review dataset to predict whether a review is **excellent** (`score_overall >= 9.0`) using hotel attributes, user profile information, and review context features.

## Business problem
Hospitality managers want to understand which guest and hotel factors are associated with very positive experiences. A predictive model can help flag patterns linked to excellent reviews and support quality improvement.

## Deliverables covered in this notebook
- Data loading and integration across multiple CSV files
- Data cleaning and feature engineering
- Exploratory data analysis (EDA)
- Predictive modeling with model comparison
- SHAP explainability
- Bias analysis across selected user groups
- Model packaging and scoring script generation

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODELS_DIR = PROJECT_ROOT / "models"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

## 1. Dataset description

The dataset is composed of three linked CSV files:

- **hotels.csv**: hotel attributes and base quality indicators
- **users.csv**: user profile attributes
- **reviews.csv**: review scores, review date, and review text

### Join keys
- `reviews.user_id` -> `users.user_id`
- `reviews.hotel_id` -> `hotels.hotel_id`

### Prediction target
The target variable in this project is:

- `excellent_review = 1` if `score_overall >= 9.0`
- `excellent_review = 0` otherwise

This turns the problem into a binary classification task.

In [ ]:
def load_csv(filename: str) -> pd.DataFrame:
    candidates = [
        DATA_DIR / filename,
        PROJECT_ROOT / filename,
        Path.cwd() / filename,
        Path.cwd() / "data" / "raw" / filename,
        Path.cwd().parent / "data" / "raw" / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            print(f"Loading {filename} from: {candidate}")
            return pd.read_csv(candidate)
    raise FileNotFoundError(f"Could not find {filename}. Checked: {candidates}")

hotels = load_csv("hotels.csv")
users = load_csv("users.csv")
reviews = load_csv("reviews.csv")

print("\nShapes")
print("Hotels:", hotels.shape)
print("Users:", users.shape)
print("Reviews:", reviews.shape)

summary_df = pd.DataFrame({
    "file": ["hotels.csv", "users.csv", "reviews.csv"],
    "rows": [len(hotels), len(users), len(reviews)],
    "columns": [hotels.shape[1], users.shape[1], reviews.shape[1]]
})
display(summary_df)

In [ ]:
print("Hotels columns:")
print(hotels.columns.tolist())
print("\nUsers columns:")
print(users.columns.tolist())
print("\nReviews columns:")
print(reviews.columns.tolist())

In [ ]:
df = (
    reviews
    .merge(users, on="user_id", how="left")
    .merge(hotels, on="hotel_id", how="left", suffixes=("_user", "_hotel"))
)

df = df.rename(columns={
    "country_user": "user_country",
    "country_hotel": "hotel_country"
})

print("Merged shape:", df.shape)
display(df.head())
print("\nMissing values after merge:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

## 2. Data cleaning and feature engineering

The dataset is cleaned and transformed into a modeling table at the **review level**, meaning each row corresponds to one review.

### Leakage handling
The following variables are excluded from the model because they are direct component scores of the overall rating and would create target leakage:

- `score_cleanliness`
- `score_comfort`
- `score_facilities`
- `score_location`
- `score_staff`
- `score_value_for_money`
- `score_overall`

### Additional engineered features
- `review_year`
- `review_month`
- `review_quarter`
- `membership_days`
- `review_text_length`
- `review_word_count`

In [ ]:
df = df.copy()

df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")
df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce")

df["excellent_review"] = (df["score_overall"] >= 9.0).astype(int)
df["review_year"] = df["review_date"].dt.year
df["review_month"] = df["review_date"].dt.month
df["review_quarter"] = df["review_date"].dt.quarter
df["membership_days"] = (df["review_date"] - df["join_date"]).dt.days
df["review_text_length"] = df["review_text"].fillna("").str.len()
df["review_word_count"] = df["review_text"].fillna("").str.split().str.len()

print("Duplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
display(df["excellent_review"].value_counts().rename("count").to_frame())
display(df["excellent_review"].value_counts(normalize=True).rename("proportion").to_frame())

print("\nData types snapshot:")
display(df.dtypes.to_frame("dtype").head(20))

In [ ]:
leakage_cols = [
    "score_cleanliness",
    "score_comfort",
    "score_facilities",
    "score_location",
    "score_staff",
    "score_value_for_money",
    "score_overall"
]

drop_cols = [
    "review_id",
    "user_id",
    "hotel_id",
    "review_date",
    "join_date",
    "review_text",
    "excellent_review"
] + leakage_cols

feature_cols = [col for col in df.columns if col not in drop_cols]
constant_cols = [col for col in feature_cols if df[col].nunique(dropna=False) <= 1]
feature_cols = [col for col in feature_cols if col not in constant_cols]

print("Dropped constant columns:", constant_cols)
print("Final feature columns:")
print(feature_cols)

X = df[feature_cols].copy()
y = df["excellent_review"].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("\nNumeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

## 3. Exploratory data analysis

In [ ]:
target_counts = df["excellent_review"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(target_counts.index.astype(str), target_counts.values)
plt.title("Target Distribution: Excellent vs Not Excellent Review")
plt.xlabel("Excellent Review")
plt.ylabel("Count")
plt.show()

In [ ]:
excellent_by_gender = df.groupby("user_gender", as_index=False)["excellent_review"].mean()

plt.figure(figsize=(8, 4))
plt.bar(excellent_by_gender["user_gender"], excellent_by_gender["excellent_review"])
plt.title("Excellent Review Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Excellent Review Rate")
plt.ylim(0, 1)
plt.show()

In [ ]:
excellent_by_traveller = (
    df.groupby("traveller_type", as_index=False)["excellent_review"]
      .mean()
      .sort_values("excellent_review", ascending=False)
)

plt.figure(figsize=(10, 5))
plt.bar(excellent_by_traveller["traveller_type"], excellent_by_traveller["excellent_review"])
plt.title("Excellent Review Rate by Traveller Type")
plt.xlabel("Traveller Type")
plt.ylabel("Excellent Review Rate")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.show()

In [ ]:
excellent_by_city = (
    df.groupby("city", as_index=False)["excellent_review"]
      .mean()
      .sort_values("excellent_review", ascending=False)
)

plt.figure(figsize=(10, 5))
plt.bar(excellent_by_city["city"], excellent_by_city["excellent_review"])
plt.title("Excellent Review Rate by City")
plt.xlabel("City")
plt.ylabel("Excellent Review Rate")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.show()

In [ ]:
trend = (
    df.groupby(["review_year", "review_month"], as_index=False)["excellent_review"]
      .mean()
      .sort_values(["review_year", "review_month"])
)
trend["year_month"] = trend["review_year"].astype(str) + "-" + trend["review_month"].astype(str).str.zfill(2)

plt.figure(figsize=(12, 4))
plt.plot(trend["year_month"], trend["excellent_review"], marker="o")
plt.title("Excellent Review Rate Over Time")
plt.xlabel("Year-Month")
plt.ylabel("Excellent Review Rate")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.show()

In [ ]:
numeric_summary = df[numeric_features + ["excellent_review"]].describe().T
display(numeric_summary)

## 4. Model development

Three baseline models are compared:

- Logistic Regression
- Random Forest
- Gradient Boosting

The final model is selected using **ROC-AUC** as the primary criterion, supported by accuracy, precision, recall, and F1-score.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred_tmp = pipe.predict(X_test)
    y_prob_tmp = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred_tmp),
        "precision": precision_score(y_test, y_pred_tmp),
        "recall": recall_score(y_test, y_pred_tmp),
        "f1": f1_score(y_test, y_pred_tmp),
        "roc_auc": roc_auc_score(y_test, y_prob_tmp)
    })

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, "model"]
best_pipeline = fitted_pipelines[best_model_name]

print("Best model:", best_model_name)

In [ ]:
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0,1], ["Pred 0", "Pred 1"])
plt.yticks([0,1], ["True 0", "True 1"])
for r in range(cm.shape[0]):
    for c in range(cm.shape[1]):
        plt.text(c, r, cm[r, c], ha="center", va="center", color="black")
plt.colorbar()
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f"ROC Curve - {best_model_name}")
plt.show()

In [ ]:
preprocessor_fitted = best_pipeline.named_steps["preprocessor"]
model_fitted = best_pipeline.named_steps["model"]
feature_names = preprocessor_fitted.get_feature_names_out()

if hasattr(model_fitted, "feature_importances_"):
    importance_values = model_fitted.feature_importances_
elif hasattr(model_fitted, "coef_"):
    importance_values = np.abs(model_fitted.coef_[0])
else:
    importance_values = None

if importance_values is not None:
    importance_df = (
        pd.DataFrame({"feature": feature_names, "importance": importance_values})
          .sort_values("importance", ascending=False)
          .head(15)
    )
    display(importance_df)

    plt.figure(figsize=(10, 6))
    plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
    plt.title(f"Top 15 Model Drivers - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()

## 5. SHAP explainability

The notebook uses a model-specific SHAP explainer:

- **LinearExplainer** for Logistic Regression
- **TreeExplainer** for Random Forest / Gradient Boosting

A small sample is used for efficiency.

In [ ]:
X_sample = X_test.sample(n=min(500, len(X_test)), random_state=42)
X_sample_transformed = preprocessor_fitted.transform(X_sample)

# background sample for SHAP
background = X_train.sample(n=min(500, len(X_train)), random_state=42)
background_transformed = preprocessor_fitted.transform(background)

if hasattr(model_fitted, "coef_"):
    explainer = shap.LinearExplainer(model_fitted, background_transformed)
    shap_values = explainer(X_sample_transformed)
    shap_values_matrix = shap_values.values
elif hasattr(model_fitted, "feature_importances_"):
    explainer = shap.TreeExplainer(model_fitted)
    shap_values_raw = explainer.shap_values(X_sample_transformed)
    if isinstance(shap_values_raw, list):
        shap_values_matrix = shap_values_raw[1]
    else:
        shap_values_matrix = shap_values_raw
        if len(np.array(shap_values_matrix).shape) == 3:
            shap_values_matrix = shap_values_matrix[:, :, 1]
else:
    raise ValueError("Unsupported model type for SHAP.")

print("SHAP matrix shape:", np.array(shap_values_matrix).shape)

In [ ]:
shap.summary_plot(
    shap_values_matrix,
    X_sample_transformed,
    feature_names=feature_names,
    max_display=15
)

In [ ]:
shap.summary_plot(
    shap_values_matrix,
    X_sample_transformed,
    feature_names=feature_names,
    plot_type="bar",
    max_display=15
)

## 6. Bias analysis

Bias analysis compares model performance across:
- `user_gender`
- `traveller_type`

Metrics reported:
- accuracy
- precision
- recall
- F1-score
- false positive rate
- false negative rate

In [ ]:
bias_df = X_test.copy()
bias_df["y_true"] = y_test.values
bias_df["y_pred"] = y_pred
bias_df["y_prob"] = y_prob

def compute_group_metrics(data: pd.DataFrame, group_col: str, min_count: int = 50) -> pd.DataFrame:
    rows = []
    for group_value, group_data in data.groupby(group_col):
        if len(group_data) < min_count:
            continue

        yt = group_data["y_true"]
        yp = group_data["y_pred"]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

        rows.append({
            "group": group_value,
            "count": len(group_data),
            "accuracy": accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "f1": f1_score(yt, yp, zero_division=0),
            "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else 0,
            "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0
        })

    return pd.DataFrame(rows).sort_values("count", ascending=False)

gender_metrics = compute_group_metrics(bias_df, "user_gender")
traveller_metrics = compute_group_metrics(bias_df, "traveller_type")

print("Bias analysis by gender")
display(gender_metrics)

print("Bias analysis by traveller type")
display(traveller_metrics)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(gender_metrics["group"], gender_metrics["recall"])
plt.title("Recall by Gender")
plt.xlabel("Gender")
plt.ylabel("Recall")
plt.ylim(0, 1)
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(traveller_metrics["group"], traveller_metrics["recall"])
plt.title("Recall by Traveller Type")
plt.xlabel("Traveller Type")
plt.ylabel("Recall")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.show()

## 7. Deployment preparation

In [ ]:
model_path = MODELS_DIR / "best_model.joblib"
features_path = MODELS_DIR / "model_features.joblib"
sample_input_path = MODELS_DIR / "sample_scoring_input.csv"

joblib.dump(best_pipeline, model_path)
joblib.dump(feature_cols, features_path)

sample_input = X_test.head(5).copy()
sample_output = sample_input.copy()
sample_output["predicted_label"] = best_pipeline.predict(sample_input)
sample_output["predicted_probability"] = best_pipeline.predict_proba(sample_input)[:, 1]
sample_input.to_csv(sample_input_path, index=False)

print("Saved model to:", model_path)
print("Saved feature list to:", features_path)
print("Saved sample scoring input to:", sample_input_path)
display(sample_output)

In [ ]:
score_script = """
from pathlib import Path
import argparse
import pandas as pd
import joblib

BASE_DIR = Path(__file__).resolve().parents[1]
DEFAULT_MODEL_PATH = BASE_DIR / "models" / "best_model.joblib"

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Path to input CSV file")
    parser.add_argument("--output", default="predictions.csv", help="Path to output CSV file")
    parser.add_argument("--model", default=str(DEFAULT_MODEL_PATH), help="Path to saved model")
    args = parser.parse_args()

    model = joblib.load(args.model)
    input_df = pd.read_csv(args.input)

    output_df = input_df.copy()
    output_df["predicted_label"] = model.predict(input_df)
    output_df["predicted_probability"] = model.predict_proba(input_df)[:, 1]

    output_df.to_csv(args.output, index=False)
    print(f"Predictions saved to: {args.output}")

if __name__ == "__main__":
    main()
""".strip()

score_script_path = DEPLOYMENT_DIR / "score.py"
score_script_path.write_text(score_script, encoding="utf-8")
print("Wrote scoring script to:", score_script_path)

## 8. Business insights and recommendations

Use the outputs above to summarize the strongest findings. Good points to include in your report or slides:

- Which features are the strongest positive or negative drivers of an excellent review
- Whether performance differs across traveller types or gender groups
- Which hotels, cities, or user segments show stronger excellent-review rates
- How the model can help support quality improvement and guest experience monitoring

## 9. Dashboard and presentation checklist

### Dashboard pages to build
1. **Executive overview**
   - total reviews
   - excellent review rate
   - average overall score
   - reviews by city / hotel

2. **Guest satisfaction drivers**
   - excellent review rate by traveller type
   - excellent review rate by gender
   - excellent review rate over time

3. **Hotel performance**
   - review volume by hotel
   - excellent review rate by hotel
   - comparison of hotel base quality indicators

4. **Model and fairness**
   - confusion matrix
   - model performance metrics
   - top features / SHAP summary
   - recall by group

### Slides to prepare
- project overview
- dataset and joins
- cleaning and feature engineering
- EDA highlights
- model comparison
- SHAP explainability
- bias analysis
- deployment and scoring script
- dashboard walkthrough
- final recommendations

## 10. Conclusion

This notebook delivers a complete review-level analytics workflow for the hotel dataset. It integrates the source files, builds a binary classification target, compares baseline models, explains the final model using SHAP, checks for performance gaps across selected groups, and saves deployment-ready model artifacts.